# Projet Kayak — pipeline météo → scraping → S3 → RDS → visualisation

### Setup — imports, secrets, connexions AWS / RDS

In [1]:

# !pip install psycopg2-binary

from dotenv import load_dotenv
import os
import time
from collections import defaultdict
from datetime import datetime, timedelta

import requests
import pandas as pd
import numpy as np
import boto3
from sqlalchemy import create_engine, text, Column, Integer, String, Float, ForeignKey
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import relationship
import plotly.express as px

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

load_dotenv(override=True)  # lit le fichier .env et charge les variables dans l'environnement


True

In [2]:
# --- Session AWS (S3) ---
session = boto3.Session(
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name="eu-west-3"  # Paris — ajuste si ton compte AWS est configuré ailleurs
)
s3 = session.resource("s3")


In [3]:
# --- Connexion RDS (PostgreSQL) ---
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Test de connexion — une requête qui ne dépend d'aucune table existante
with engine.connect() as conn:
    resultat = conn.execute(text("SELECT 1"))
    print("Connexion RDS réussie :", resultat.fetchone())


Connexion RDS réussie : (1,)


### OpenWeather — météo dynamique + scoring

In [4]:
api_key = os.getenv("API_KEY")

POIDS_TEMP = 4
POIDS_PLUIE = 40
TEMP_IDEALE = 25


def recuperer_previsions(ville, lat, lon):
    """Récupère les prévisions 5 jours / 3h et agrège temp/pluie par jour."""
    url = (
        f"https://api.openweathermap.org/data/2.5/forecast"
        f"?lat={lat}&lon={lon}&units=metric&lang=fr&appid={api_key}"
    )
    response = requests.get(url, timeout=10)
    if response.status_code != 200:
        print(f"⚠ Erreur pour {ville} : {response.status_code}")
        return None

    data = response.json()
    temps_par_jour = defaultdict(lambda: {"temps": [], "pops": []})
    for creneau in data["list"]:
        date = creneau["dt_txt"].split(" ")[0]
        temps_par_jour[date]["temps"].append(creneau["main"]["temp"])
        temps_par_jour[date]["pops"].append(creneau.get("pop", 0))

    jours = []
    for date in sorted(temps_par_jour.keys())[:5]:
        temperatures = temps_par_jour[date]["temps"]
        pops = temps_par_jour[date]["pops"]
        jours.append({
            "date": date,
            "temp_max": round(max(temperatures), 1),
            "temp_moy": round(sum(temperatures) / len(temperatures), 1),
            "pop_moy": round(sum(pops) / len(pops), 2),
        })
    return jours


def calcul_score(row):
    ecart_temp = abs(row["temp_moy"] - TEMP_IDEALE)
    penalite_pluie = row["pop_moy"] * POIDS_PLUIE
    return round(100 - ecart_temp * POIDS_TEMP - penalite_pluie, 1)


def get_weather_scores(cities_df):
    """
    Récupère la météo J+1 à J+5 pour chaque ville, calcule le score
    'beau temps', renvoie (meteo_detail, meteo_scores) frais du jour.
    """
    lignes_detail = []

    for _, row in cities_df.iterrows():
        jours = recuperer_previsions(row["ville"], row["lat"], row["lon"])
        if not jours:
            continue
        for jour in jours:
            lignes_detail.append({
                "city_id": row["city_id"], "ville": row["ville"],
                "lat": row["lat"], "lon": row["lon"], **jour
            })
        time.sleep(1)  # respecter le rate limit OpenWeather (60 appels/min en gratuit)

    meteo_detail = pd.DataFrame(lignes_detail)

    agg = (
        meteo_detail
        .groupby(["city_id", "ville", "lat", "lon"], as_index=False)[["temp_moy", "pop_moy"]]
        .mean()
    )
    agg["score"] = agg.apply(calcul_score, axis=1)
    meteo_scores = agg.sort_values("score", ascending=False).reset_index(drop=True)

    # les CSV restent utiles comme trace/log du run, mais ne sont plus relus comme source
    meteo_detail.to_csv("meteo_detail.csv", index=False)
    meteo_scores.to_csv("meteo_scores.csv", index=False)

    return meteo_detail, meteo_scores


### Booking — scraping Selenium

In [5]:
def scrape_city(driver, city_name, city_id, city_lat, city_lon, checkin, checkout):

    # 1. construire l'URL avec city_name
    url = (
        f"https://www.booking.com/searchresults.fr.html"
        f"?ss={city_name}&checkin={checkin}&checkout={checkout}"
        f"&group_adults=2&order=bayesian_review_score"
    )
    # 2. y aller
    driver.get(url)

    # 3. fermer les popups (cookies + Genius) — tolérant si absentes
    try:
        WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "[aria-label*='Ignorer']"))
        ).click()
    except TimeoutException:
        pass

    try:
        WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
        ).click()
    except TimeoutException:
        pass

    # 4. attendre les cartes, puis les récupérer
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "[data-testid='property-card']"))
        )
        cartes = driver.find_elements(By.CSS_SELECTOR, "[data-testid='property-card']")
    except TimeoutException:
        cartes = []

    # 5. boucle 1 : nom, note, lien → hotels
    hotels = []
    for carte in cartes[:5]:
        try:
            nom = carte.find_element(By.CSS_SELECTOR, "[data-testid='title']").text
        except NoSuchElementException:
            nom = None

        try:
            note_brute = carte.find_element(By.CSS_SELECTOR, "[data-testid='review-score']").text
            note = float(note_brute.split("\n")[1].replace(",", "."))
        except (NoSuchElementException, IndexError, ValueError):
            note = None

        try:
            lien = carte.find_element(By.CSS_SELECTOR, "a[data-testid='title-link']").get_attribute("href")
            lien = lien.split("?")[0]
        except NoSuchElementException:
            lien = None

        hotels.append({"city_id": city_id, "ville": city_name, "lat": city_lat, "lon": city_lon,
               "hotel": nom, "note": note, "lien": lien})

    # 6. boucle 2 : ouvrir chaque fiche → description
    for h in hotels:
        if h["lien"] is None:
            h["description"] = None
            continue
        driver.get(h["lien"])
        try:
            h["description"] = driver.find_element(
                By.CSS_SELECTOR, "[data-testid='property-description']"
            ).text
        except NoSuchElementException:
            h["description"] = None

    return hotels


### Exécution — météo live → top 5 → scraping Booking

In [6]:
# ouverture du navigateur (une seule fois)
driver = webdriver.Chrome()

cities_df = pd.read_csv("villeKayak.csv")

meteo_detail, meteo_scores = get_weather_scores(cities_df)
top5 = meteo_scores.head(5)
print(top5[["ville", "score"]])

checkin = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")
checkout = (datetime.now() + timedelta(days=5)).strftime("%Y-%m-%d")

resultats_complets = []
for _, row in top5.iterrows():
    print(f"Scraping {row['ville']} (score météo : {row['score']})...")
    try:
        hotels_ville = scrape_city(
            driver, row['ville'], row['city_id'], row['lat'], row['lon'],
            checkin, checkout
        )
        resultats_complets.extend(hotels_ville)
        pd.DataFrame(resultats_complets).to_csv("hotels_booking.csv", index=False)
    except Exception as e:
        print(f"Erreur sur {row['ville']} : {e}")
        continue


                          ville  score
0             Mont Saint Michel   95.4
1                       Bayonne   95.1
2                      Le Havre   94.8
3  Chateau du Haut Koenigsbourg   94.5
4                   La Rochelle   94.2
Scraping Mont Saint Michel (score météo : 95.4)...
Scraping Bayonne (score météo : 95.1)...
Scraping Le Havre (score météo : 94.8)...
Scraping Chateau du Haut Koenigsbourg (score météo : 94.5)...
Scraping La Rochelle (score météo : 94.2)...


### S3 — fusion et envoi du CSV enrichi

In [7]:
hotels = pd.read_csv("hotels_booking.csv")

enrichi = hotels.merge(
    top5[["city_id", "score"]],
    on="city_id",
    how="left"
)
enrichi.to_csv("destinations_enrichies.csv", index=False)

csv_bytes = enrichi.to_csv(index=False)
bucket_name = "kayak91"

bucket = s3.Bucket(bucket_name)
bucket.put_object(Key="destinations_enrichies.csv", Body=csv_bytes)

print(f"Nombre de lignes envoyées : {len(enrichi)}")
print(f"Envoyé : destinations_enrichies.csv → s3://{bucket_name}/")


Nombre de lignes envoyées : 24
Envoyé : destinations_enrichies.csv → s3://kayak91/


### RDS — schéma normalisé (villes / hotels)

In [8]:
Base = declarative_base()

# --- Table "villes" : une ligne par ville, avec son score météo -----------
class Ville(Base):
    __tablename__ = "villes"

    city_id = Column(Integer, primary_key=True)
    ville = Column(String)
    lat = Column(Float)
    lon = Column(Float)
    score = Column(Float)

    hotels = relationship("Hotel", back_populates="ville_rel")


# --- Table "hotels" : une ligne par hôtel, reliée à sa ville --------------
class Hotel(Base):
    __tablename__ = "hotels"

    id = Column(Integer, primary_key=True, autoincrement=True)
    city_id = Column(Integer, ForeignKey("villes.city_id"))
    hotel = Column(String)
    note = Column(Float)
    lien = Column(String)
    description = Column(String)

    ville_rel = relationship("Ville", back_populates="hotels")

Base.metadata.create_all(engine)
print("Tables 'villes' et 'hotels' créées (ou déjà existantes).")


Tables 'villes' et 'hotels' créées (ou déjà existantes).


C:\Users\Mfumu\AppData\Local\Temp\ipykernel_39360\3262751064.py:1: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [9]:
with engine.connect() as conn:
    conn.execute(text("TRUNCATE TABLE hotels, villes RESTART IDENTITY CASCADE"))
    conn.commit()
villes_uniques = enrichi[["city_id", "ville", "lat", "lon", "score"]].drop_duplicates()
villes_uniques.to_sql("villes", engine, if_exists="append", index=False)

hotels_seuls = enrichi[["city_id", "hotel", "note", "lien", "description"]]
hotels_seuls.to_sql("hotels", engine, if_exists="append", index=False)

print(f"{len(villes_uniques)} villes et {len(hotels_seuls)} hôtels insérés.")


5 villes et 24 hôtels insérés.


In [10]:
with engine.connect() as conn:
    resultat = conn.execute(text("""
        SELECT v.ville, v.score, h.hotel, h.note
        FROM villes v
        JOIN hotels h ON v.city_id = h.city_id
        ORDER BY v.score DESC, h.note DESC
    """))
    for ligne in resultat:
        print(ligne)


('Mont Saint Michel', 95.4, 'Villa MONTGOMMERY', 10.0)
('Mont Saint Michel', 95.4, 'Ermitage - Mont-Saint-Michel', 8.7)
('Mont Saint Michel', 95.4, 'Appart Hôtel de la baie', 8.5)
('Mont Saint Michel', 95.4, 'Le vieux logis', 2.0)
('Bayonne', 95.1, 'Hôtel du Palais Biarritz, in The Unbound Collection by Hyatt', 9.5)
('Bayonne', 95.1, 'OKKO Hotels Bayonne Centre', 8.3)
('Bayonne', 95.1, 'Campanile NATURE - Bayonne', 6.9)
('Bayonne', 95.1, 'Hotel Bistrot FINE', 6.9)
('Bayonne', 95.1, 'Maubec YourHostHelper', 4.2)
('Le Havre', 94.8, 'Maison Douce Époque - Deauville', 9.5)
('Le Havre', 94.8, 'Hôtel Barrière Le Normandy', 8.7)
('Le Havre', 94.8, 'JOST Hôtel Le Havre Centre Gare', 8.5)
('Le Havre', 94.8, 'JOST Auberge de Jeunesse Le Havre Centre Gare', 8.3)
('Le Havre', 94.8, 'The Originals Boutique, Hôtel Le Marignan,Le Havre Centre Gare', 8.0)
('Chateau du Haut Koenigsbourg', 94.5, 'Hôtel Le Mittelwihr', 8.8)
('Chateau du Haut Koenigsbourg', 94.5, 'Hôtel De La Couronne', 8.7)
('Chateau du 

### Visualisation — cartes Plotly

In [11]:
top5_villes = pd.read_sql("SELECT * FROM villes ORDER BY score DESC", engine)

fig_destinations = px.scatter_mapbox(
    top5_villes,
    lat="lat",
    lon="lon",
    color="score",
    size="score",
    hover_name="ville",
    hover_data={"score": True, "lat": False, "lon": False},
    color_continuous_scale="Sunset",
    zoom=4.5,
    center={"lat": 46.6, "lon": 2.5},
    mapbox_style="open-street-map",
    title="Top 5 des destinations — score météo (J+1 à J+5)"
)
fig_destinations.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig_destinations.show()


C:\Users\Mfumu\AppData\Local\Temp\ipykernel_39360\1450055580.py:3: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_destinations = px.scatter_mapbox(


In [12]:
top20_hotels = pd.read_sql("""
    SELECT h.hotel, h.note, h.lien, v.ville, v.lat, v.lon, v.score
    FROM hotels h
    JOIN villes v ON h.city_id = v.city_id
    ORDER BY h.note DESC
    LIMIT 20
""", engine)

fig_hotels = px.scatter_mapbox(
    top20_hotels,
    lat="lat",
    lon="lon",
    color="note",
    size="note",
    hover_name="hotel",
    hover_data={"ville": True, "note": True, "lat": False, "lon": False},
    color_continuous_scale="Viridis",
    zoom=4.5,
    center={"lat": 46.6, "lon": 2.5},
    mapbox_style="open-street-map",
    title="Top 20 des hôtels — meilleures notes clients"
)
fig_hotels.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig_hotels.show()


C:\Users\Mfumu\AppData\Local\Temp\ipykernel_39360\2230772239.py:9: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

